## Claude - Opus 5.5 summarize

### **Preprocessing & alignment**

- **RevertSam**: strips alignment info from a BAM to produce an unmapped BAM (uBAM), keeping read groups for clean realignment.

- **samtools fastq -n -0 … | bwa-mem2 mem … | samtools sort**: converts the uBAM to interleaved FASTQ, aligns to GRCh38 with bwa-mem2 using the read-group tag, and coordinate-sorts the output.
- **MarkDuplicates**: flags PCR/optical duplicate reads so they are ignored downstream.
- **BaseRecalibrator**: models systematic base-quality errors against known variant sites (BQSR step 1).
- **ApplyBQSR**: applies that model to rewrite base qualities in the BAM (BQSR step 2).

### **QC**

- **mosdepth -t 4**: fast per-region and genome-wide coverage depth, using 4 threads.

- **CollectWgsMetrics**: WGS coverage stats (mean depth, % bases ≥ X, filtered bases).
- **CollectInsertSizeMetrics**: fragment insert-size distribution from paired reads.
- **verifybamid2 --SVDPrefix**: estimates DNA cross-sample contamination (FREEMIX) and ancestry PCs from a reference SVD panel.
- **somalier extract**: pulls genotypes at informative sites for sample identity, relatedness, and tumor–normal pairing checks.

**mosdepth -t 4**
Computes read depth per base, then summarizes it per chromosome, per BED region (for example exome targets or 500 bp windows), and as cumulative distributions. It is much faster than `samtools depth`, and `-t 4` adds 4 decompression threads. It writes `*.mosdepth.summary.txt`, `*.regions.bed.gz`, and `*.global.dist.txt`.
```
chrom   length      bases        mean   min  max
chr1    248956422   8712475301   35.00  0    4012
chrX    156040895   2652695215   17.00  0    1890   # ~half depth → male sample
total   3099734149  105390961066 34.00  0    9870
```

**CollectWgsMetrics**
This is Picard's WGS coverage report. It gives mean and median depth over the genome territory, the fraction of bases excluded (low MAPQ, duplicates, low base quality, overlapping mates), and the percentage of the genome reaching ≥1X, 10X, 20X, 30X and so on. It is the standard check that a sample reached its target depth.
```
GENOME_TERRITORY  MEAN_COVERAGE  MEDIAN_COVERAGE  PCT_EXC_DUPE  PCT_EXC_MAPQ  PCT_10X  PCT_20X  PCT_30X
2745186691        32.4           33               0.082         0.031         0.981    0.962    0.611
```

**CollectInsertSizeMetrics**
Measures the DNA fragment length from properly paired reads, reported as median, mean, SD, and pair orientation, plus a histogram PDF. It flags library-prep problems (over- or under-fragmentation) and is used by SV callers. cfDNA shows a characteristic peak near 167 bp.
```
MEDIAN_INSERT_SIZE  MODE_INSERT_SIZE  MEAN_INSERT_SIZE  STANDARD_DEVIATION  READ_PAIRS  PAIR_ORIENTATION
412                 398               418.7             102.3               412553120   FR
```

**verifybamid2 --SVDPrefix**
Estimates whether the BAM contains DNA from another person. It models allele fractions at common SNPs using a population SVD panel (1000G, via `--SVDPrefix`), so no external genotypes are needed. The key output is **FREEMIX**, the estimated contamination fraction: below about 0.02–0.03 is usually acceptable. It also reports the sample's ancestry PCs in `*.selfSM` and `*.Ancestry`.
```
#SEQ_ID  RG  CHIP_ID  #SNPS  #READS    AVG_DP  FREEMIX   FREELK1   FREELK0
HP01_N   NA  NA       9885   312004    31.56   0.00412   3.21e+04  3.22e+04
```

**somalier extract**
Genotypes a small panel of about 17k highly informative common SNPs from each BAM or CRAM into a compact binary `*.somalier` file, which is not human-readable. You then run `somalier relate` on all the files to get pairwise relatedness. Use it to confirm that tumor and normal come from the same patient, to catch sample swaps, and to find duplicates or relatives. Expect relatedness ≈ 1.0 for same-individual pairs and ≈ 0 for unrelated samples.
```
#sample_a  sample_b  relatedness  ibs0  ibs2   hom_concordance  n
HP01_N     HP01_H    0.97         2     11840  0.95             12511   # same patient ✓
HP01_N     HP02_N    0.01         1420  4102   -0.05            12488   # unrelated ✓
```

### **Germline calling**

- **HaplotypeCaller**: per-sample local-reassembly variant calling, typically to GVCF.

- **GenomicsDBImport**: consolidates many per-sample GVCFs into a GenomicsDB workspace.
- **GenotypeGVCFs**: joint genotyping across the cohort from GenomicsDB.
- **GatherVcfs**: concatenates per-interval VCF shards into one VCF.
- **bcftools index -t**: builds a tabix (.tbi) index for the germline VCF.


### **Somatic calling**

- **Mutect2**: somatic SNV/indel caller for tumor vs. normal; also used on normals to build the PoN and in mitochondrial mode.

- **CreateSomaticPanelOfNormals**: aggregates normal-sample calls into a PoN that filters recurrent artifacts and germline noise.
- **MergeMutectStats**: combines per-shard Mutect2 `.stats` files so filtering is correct.
- **FilterMutectCalls**: applies Mutect2's probabilistic filters (contamination, orientation, strand, PoN, and others) to raw calls.

### Focused on algorithms, what is the difference between Germline Calling and Somatic Calling?

Both callers share the same front end: find active regions, locally reassemble haplotypes with a de Bruijn graph, then score each read against each haplotype with a PairHMM to get P(read | haplotype). They differ in the statistical model applied to those likelihoods, because they ask different questions.

**Germline (HaplotypeCaller → GenotypeGVCFs)**

The question is: *which genotype does this individual carry?*

The model assumes a fixed **ploidy** (2). The only possible allele fractions are 0, 0.5 or 1, so the genotype space is discrete (0/0, 0/1, 1/1). For a genotype made of haplotypes h₁ and h₂:

P(reads | G) = ∏ᵣ [ ½·P(r|h₁) + ½·P(r|h₂) ]

Bayes' rule with a heterozygosity prior (about 1/1000) then gives a posterior per genotype, reported as GT, PL and GQ.

It is a **cohort** approach. The GVCF also stores reference confidence at non-variant sites, so joint genotyping (GenomicsDB + GenotypeGVCFs) can pool evidence across samples and use cohort allele frequencies as priors. That rescues low-coverage hets and distinguishes "hom-ref" from "no data."

Errors are handled by VQSR or hard filters on site-level annotations (QD, FS, MQ, and so on).

**Somatic (Mutect2 → FilterMutectCalls)**

The question is: *is a non-germline variant present at all, at any fraction?*

There is **no ploidy assumption**. The allele fraction f is continuous and unknown, because it depends on tumor purity, subclonality and copy number:

P(reads | f) = ∏ᵣ [ f·P(r|alt) + (1−f)·P(r|ref) ]

f is integrated over a Beta prior. The key statistic is the **TLOD**, the log-odds of "variant present at some f" against "reference plus sequencing error." This lets it detect variants at 1–5% VAF, where a diploid model would call 0/0.

It is a **tumor-vs-normal, per-patient** approach:
- **NLOD** tests that the allele is absent in the matched normal.
- The **gnomAD allele-frequency prior** down-weights likely germline variants, which matters most with no normal or with tumor-in-normal contamination.
- The **Panel of Normals** removes recurrent technical artifacts seen across unrelated normals.

Errors are handled by a **probabilistic filtering** stage, FilterMutectCalls. It combines per-site probabilities for:
- contamination (CalculateContamination)
- orientation bias from FFPE or oxoG damage (LearnReadOrientationModel)
- strand bias, mapping quality and clustered events
- a learned somatic allele-fraction spectrum

It then chooses the threshold that maximizes the expected F-score.

**Summary**

| | Germline | Somatic |
|---|---|---|
| Allele fraction | Discrete (0, ½, 1) | Continuous f ∈ (0,1] |
| Core output | Genotype (GT/GQ) | Presence + AF (TLOD) |
| Main enemy | Missing hets, genotyping errors | Germline leakage, low-VAF artifacts |
| Evidence pooling | Across cohort (joint calling) | Tumor vs. matched normal + PoN |
| Priors | Heterozygosity, cohort AF | gnomAD AF (germline), somatic AF spectrum |

This is also why your **mitochondrial** workflow uses Mutect2 (`--mitochondria-mode`) and not HaplotypeCaller. mtDNA has hundreds of copies with **heteroplasmy**, so allele fractions are continuous, exactly the somatic-model assumption.

### **Mitochondrial workflow**

- **PrintReads**: extracts a subset of reads, here chrM, into a new BAM.

- **samtools collate -Ou**: groups reads by name (uncompressed output) so pairs stay together before FASTQ conversion and realignment.
- **bwa mem -K 100000000**: realigns chrM reads to the standard and shifted mito references, with a fixed batch size for reproducibility.
- **LiftoverVcf**: maps calls made on the shifted chrM reference back to standard coordinates, which handles the control-region breakpoint.
- **MergeVcfs**: merges the non-shifted and lifted-over shifted VCFs into one callset.
- **VariantFiltration**: applies hard filters or tags variants, for example blacklisted sites or low heteroplasmy.
- **LeftAlignAndTrimVariants**: normalizes indels by left-aligning and trimming, and can split multiallelics.
- **haplocheck**: detects mtDNA contamination from haplogroup mixture in heteroplasmic calls.

### Manta and Strelka

Manta and Strelka are complementary bioinformatics tools developed by Illumina for calling genetic variants from next-generation sequencing data.

How They Work Together

- Manta (Structural Variant Caller) detects structural variants (SVs) and large insertions/deletions (indels) from mapped paired-end reads. It also generates a candidate list of small indels.

- Strelka (Small Variant Caller / Strelka2) identifies germline and somatic small variants, specifically single nucleotide variants (SNVs) and small indels.

- Pipeline Integration: Illumina best practices recommend running Manta first and feeding its candidate small indels into Strelka (via the --indelCandidates option) to maximize sensitivity and achieve complete, seamless coverage across all indel and variant sizes.